# MultiAgentic RAG Using Langchain #

In [97]:
import warnings 
warnings.filterwarnings('ignore')

from langchain_community.document_loaders import PyPDFLoader 

loader = PyPDFLoader('economics_research_reference.pdf')
pages = loader.load()


In [98]:
# Building chunks 
from langchain_text_splitters import RecursiveCharacterTextSplitter 
text_spliter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=140)
texts=text_spliter.split_documents(pages)
chunks = [i.page_content for i in texts]
metadata = [i.metadata for i in texts]

print(f"Embedding {len(chunks)} chunks...")


Embedding 31 chunks...


In [99]:
# Create VectorDB 
import chromadb
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction
embedding_function = DefaultEmbeddingFunction()

client = chromadb.PersistentClient(path="./Database_VD")
collection = client.get_or_create_collection(name="clean_vectorDB",embedding_function=embedding_function)

if collection.count()==0:
    collection.add(
        documents=chunks,
        ids=[str(i) for i in range(len(chunks))],
        metadatas=metadata
    )


In [100]:
# Import models and local enviroment 
from langchain_groq import ChatGroq 
from dotenv import load_dotenv 

# Import tools 
from langchain_core.tools import tool 
from langchain_community.tools import DuckDuckGoSearchRun  

load_dotenv()
import os 

key = os.getenv('GROQ_API_KEY')
chat_model = ChatGroq(model='llama-3.1-8b-instant',api_key=key) 


In [101]:
@tool 
def calculator(exec:str):
    """ do the calculations based on the user given expression """
    try:
        return str(eval(exec))
    except Exception:
        print('provide the symbol')




@tool 
def retrival(query:str):
    """ provide the answer of user's qustions based on the given local document  """
    result = collection.query(query_texts=[query],n_results=5)
    distance = result['distances'][0]
    document = result['documents'][0]
    thresold = 1.0 

    near_docs = []
    for i , doc in zip(distance,document):
        if i < thresold :
            near_docs.append(doc)

    if not near_docs:
        return "NOT relevent Content"
    return "\n\n".join(near_docs)

@tool
def web_search(query:str):
    """ use this tool when the answer or relavent docs can not find out on the given local document """
    search = DuckDuckGoSearchRun()
    return search.run(query)


In [102]:
combine_tools=[calculator,retrival,web_search]
tool_map={t.name:t for t in combine_tools} 
tool_map


{'calculator': StructuredTool(name='calculator', description='do the calculations based on the user given expression', args_schema=<class 'langchain_core.utils.pydantic.calculator'>, func=<function calculator at 0x11e204430>),
 'retrival': StructuredTool(name='retrival', description="provide the answer of user's qustions based on the given local document", args_schema=<class 'langchain_core.utils.pydantic.retrival'>, func=<function retrival at 0x11e331310>),
 'web_search': StructuredTool(name='web_search', description='use this tool when the answer or relavent docs can not find out on the given local document', args_schema=<class 'langchain_core.utils.pydantic.web_search'>, func=<function web_search at 0x11e331a60>)}

In [103]:
calculator_agent = chat_model.bind_tools([calculator])
retrival_agent = chat_model.bind_tools([retrival])
web_agent = chat_model.bind_tools([web_search])

AGENTS={
    "cal":calculator_agent,
    "retrive":retrival_agent,
    "web":web_agent
}
# Sucessfully create agents 
AGENTS


{'cal': RunnableBinding(bound=ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x11e23c280>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x11e2f00a0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'calculator', 'description': 'do the calculations based on the user given expression', 'parameters': {'properties': {'exec': {'type': 'string'}}, 'required': ['exec'], 'type': 'object'}}}]}, config={}, config_factories=[]),
 'retrive': RunnableBinding(bound=ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x11e23c280>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x11e2f00a0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {

In [104]:
def fetch(query:str):
    prompt = f"""
    Always define user asking qustions for one tool . calculator , retrive , web_search,

    before provide answers follow the instructions :
    1. any calculation or operations use calculator . example : 23*78 only arithmetic operation either everything transfer to the retrive tool
    2. after all for every qustions go for retrive if their cant find any contextual relations then go to the web_search tool , but before that dont go the web_search tool.

    qustion : {query}
     """
    response = chat_model.invoke(prompt)
    print(response.content)
    label = response.content.strip().lower()
    for key in AGENTS:
        if key in label:
            return key 
    return 'General'


In [105]:
def run (query:str , state:dict=None):
    state = state or {"history":[]}

    label = fetch(query)

    if label=="General":
        answer = chat_model.invoke(query)
    else :
        agent_select=AGENTS[label]
        response = agent_select.invoke(query)

        if not response.tool_calls:
            answer = response.content 
        else:
            call = response.tool_calls[0]
            tool_name = tool_map[call['name']]
            answer=tool_name.invoke(call['args']) 
    
    state['history'].append(
        {
            "query": query , 
            "state": state,
            "answer" : answer
        }
    )

    return answer , state  


In [106]:
state = None 

answer_1 , state = run ('what is 78 multiply 67 then add 7?' ,state)

print(answer_1)


To solve this question, I will use the calculator tool.

First, I will multiply 78 by 67. 

Using the calculator, 78 * 67 = 5226.

Next, I will add 7 to the result.

Using the calculator, 5226 + 7 = 5233.

So, the result of 78 multiplied by 67 and then added by 7 is 5233.
5233


In [108]:
query = "what is sells?' "
label = fetch(query)
print(f"[routed to: {label}]")   


To answer your question, I will follow the instructions:

1. Since there is no calculation involved, I will not use the calculator. 
2. I will first try to retrieve the information from my knowledge base.
3. If I cannot find any relevant information, I will then proceed to the web_search tool.

Retrieving information...

According to my knowledge base, "sells" is a verb that means to offer for sale or dispose of something in exchange for money or other compensation.
[routed to: cal]
